# Communicating environments

> Communication as part of an environment: each agent observes the messages that reached it through a comm-core network.

In [ ]:
#| default_exp comm.wrapper

In [ ]:
#| hide
from nbdev.showdoc import *

`CommunicationWrapper` makes communication part of an environment: after every reset and step, each agent's
message goes through an `AgentNetwork`, and what each agent received becomes part of its observation:

| key | per agent | content |
|---|---|---|
| `messages` | `(num_agents, D)` float32 | row `j`: the message received from agent `j` (zeros if none arrived) |
| `message_mask` | `(num_agents,)` bool | which agents' messages arrived |
| `comm_bits_sent` (info) | int | bits the agent sent at this step |
| `comm_received` (info) | int | messages the agent received at this step |

The message of an agent is its own observation keys (`message_keys`, e.g. `('position',)`) or anything a
`message_fn(env, agent, obs)` returns (None: nothing sent). Messages are sent to the agent's neighbours in the
topology, which can change at every step (`topology` a function of the env, e.g. `within_range`).

As a `World` pre-wrapper (`communication(...)` makes one per env, each with its own network and channel), every
policy, dataset and evaluation sees exactly what got through: e.g. the same world model trained or evaluated with
a perfect channel, then a lossy one, without changing its code. Messages are delivered within the step they are
sent (comm-core's latency is reported, not simulated as delay).

In [ ]:
#| export
from __future__ import annotations

import copy
from typing import Any, Callable

import gymnasium as gym
import numpy as np
from gymnasium import spaces

from stable_marl.comm.core import AgentNetwork, _comm_core, message_bits, node_id

In [ ]:
#| export
def multigrid_positions(env) -> np.ndarray:
    "Positions of the agents of a MultiGrid env, (num_agents, 2)."
    return np.array([agent.state.pos for agent in env.agents], np.float64)


def within_range(
    radius: float, # Largest distance at which two agents can communicate
    positions: Callable | None = None, # positions(env) -> (num_agents, d); default `multigrid_positions`
    ord: float = np.inf # Norm of the distance (inf: Chebyshev, i.e. squares on a grid; 2: Euclidean)
) -> Callable:
    "A topology recomputed at every step (comm-core's `DistanceTopology`): links between agents at most `radius` apart."
    positions = positions or multigrid_positions
    def topology(env):
        pos = np.asarray(positions(env), np.float64)
        return _comm_core().DistanceTopology({node_id(a): tuple(p) for a, p in enumerate(pos)}, radius, ord=ord)
    return topology


class CommunicationWrapper(gym.Wrapper):
    "Agents exchange messages over a comm-core network at every step; what they received is in their observations."
    def __init__(
        self,
        env: gym.Env, # A stable-marl `MultiAgentEnv` (possibly wrapped)
        message_keys: tuple | None = None, # Observation keys each agent sends (flattened, concatenated)
        message_fn: Callable | None = None, # message_fn(env, agent, obs) -> array of `message_dim` numbers, or None
        message_dim: int | None = None, # Size of `message_fn`'s messages
        topology='full_mesh', # 'full_mesh', 'star', 'none', a comm-core `Topology`, or topology(env) at every step
        channel=None, # comm-core `Channel` (default: perfect)
        backend=None, # comm-core `CommunicationBackend` (instead of `channel`)
        protocol=None, # comm-core `Protocol` class / factory
        center: int = 0, # Center of the 'star' topology
        bits_per_element: int | None = None # Message size accounting (default: float32, 32 bits per number)
    ):
        super().__init__(env)
        if (message_keys is None) == (message_fn is None):
            raise ValueError("pass message_keys or message_fn")
        self.num_agents = env.unwrapped.num_agents
        agent_space = env.observation_space[0]
        if message_keys is not None:
            message_keys = tuple(message_keys)
            message_dim = sum(int(np.prod(agent_space[k].shape)) or 1 for k in message_keys)
        elif message_dim is None:
            raise ValueError("message_fn needs a message_dim")
        self.message_keys, self.message_fn, self.message_dim = message_keys, message_fn, int(message_dim)
        dynamic = callable(topology) and not isinstance(topology, (str, _comm_core().Topology))
        self._topology_fn = topology if dynamic else None
        self.comm = AgentNetwork(self.num_agents, 'none' if dynamic else topology, channel=channel, backend=backend,
                                 protocol=protocol, center=center, bits_per_element=bits_per_element)
        A, D = self.num_agents, self.message_dim
        self.observation_space = spaces.Dict({a: spaces.Dict({
            **env.observation_space[a].spaces,
            'messages': spaces.Box(-np.inf, np.inf, (A, D), np.float32),
            'message_mask': spaces.MultiBinary(A)}) for a in range(A)})

    def _message(self, agent: int, obs: dict):
        if self.message_keys is not None:
            return np.concatenate([np.asarray(obs[agent][k], np.float32).reshape(-1) for k in self.message_keys])
        m = self.message_fn(self.env.unwrapped, agent, obs)
        if m is None:
            return None
        m = np.asarray(m, np.float32).reshape(-1)
        if m.size != self.message_dim:
            raise ValueError(f"message_fn returned {m.size} numbers for agent {agent}, message_dim is {self.message_dim}")
        return m

    def _communicate(self, obs: dict, infos: dict):
        if self._topology_fn is not None:
            self.comm.set_topology(self._topology_fn(self.env.unwrapped))
        payloads = {a: self._message(a, obs) for a in range(self.num_agents)}
        received = self.comm.exchange(payloads)
        sent = {a: 0 for a in range(self.num_agents)}
        for r in self.comm.last_results:
            sent[int(r.transmission.sender.rsplit('_', 1)[1])] += r.size_bits
        A, D = self.num_agents, self.message_dim
        obs, infos = dict(obs), dict(infos)
        for a in range(A):
            messages, mask = np.zeros((A, D), np.float32), np.zeros(A, bool)
            for j, payload in received[a].items():
                messages[j], mask[j] = np.asarray(payload, np.float32).reshape(-1), True
            obs[a] = {**obs[a], 'messages': messages, 'message_mask': mask}
            infos[a] = {**infos.get(a, {}), 'comm_bits_sent': np.int64(sent[a]), 'comm_received': np.int64(mask.sum())}
        return obs, infos

    def reset(self, *, seed: int | None = None, options: dict | None = None):
        "Reset the env and the network (seeded with the env's `seed`); the first round of messages."
        obs, infos = self.env.reset(seed=seed, options=options)
        self.comm.reset(seed)
        return self._communicate(obs, infos)

    def step(self, actions):
        "Step the env, then one round of messages."
        obs, rewards, terminations, truncations, infos = self.env.step(actions)
        obs, infos = self._communicate(obs, infos)
        return obs, rewards, terminations, truncations, infos

    @property
    def metrics(self):
        "comm-core's metrics of the episode so far."
        return self.comm.metrics


def communication(**kwargs) -> Callable:
    """
    A `World` pre-wrapper adding a `CommunicationWrapper` (`kwargs`) to every env, each with its own copy of the
    channel / backend: `World(..., pre_wrappers=[communication(message_keys=('position',), channel=...)])`.
    """
    return lambda env: CommunicationWrapper(env, **copy.deepcopy(kwargs))

In [ ]:
show_doc(CommunicationWrapper.reset)

In [ ]:
show_doc(CommunicationWrapper.step)

### Tests

In [ ]:
# imports on their own (the docs build runs the cells with imports; the tests need comm-core)
import tempfile
from fastcore.test import test_eq, test_fail
import stable_marl as sm

In [ ]:
from comm_core.channel import AWGNChannel
env = CommunicationWrapper(sm.make('MultiGrid-FindGoal-15x15-v0', agents=3, size=9, num_obstacles=0),
                           message_keys=('position', 'direction'))
obs, infos = env.reset(seed=0)
test_eq(obs[0]['messages'].shape, (3, 3))                                   # 3 senders x (x, y, direction)
test_eq(obs[1]['message_mask'].tolist(), [True, False, True])               # everyone but itself
assert np.array_equal(obs[1]['messages'][2], np.r_[obs[2]['position'], obs[2]['direction']])
assert env.observation_space[0].contains(obs[0]) and infos[0]['comm_bits_sent'] == 2 * 3 * 32
obs, *_ , infos = env.step({a: 2 for a in range(3)})
test_eq(int(infos[2]['comm_received']), 2)
test_eq(env.metrics.transmissions, 12)
# a lossy channel and a message function; reproducible with the env seed
def lossy():
    env = CommunicationWrapper(sm.make('MultiGrid-FindGoal-15x15-v0', agents=3, size=9, num_obstacles=0),
                               message_fn=lambda env, a, obs: [a, 1.0] if a != 2 else None, message_dim=2,
                               channel=AWGNChannel(snr_db=3, packet_error_rate=0.5))
    masks = [env.reset(seed=7)[0][0]['message_mask'].tolist()]
    masks += [env.step({a: 0 for a in range(3)})[0][0]['message_mask'].tolist() for _ in range(6)]
    return masks, env.metrics
masks, m = lossy()
test_eq(lossy()[0], masks)
assert all(not mk[2] for mk in masks) and 0 < m.failed < m.transmissions    # agent 2 never sends; losses
# a range-limited topology, recomputed at every step
env = CommunicationWrapper(sm.make('MultiGrid-FindGoal-15x15-v0', agents=2, size=15, num_obstacles=0),
                           message_keys=('position',), topology=within_range(2))
obs, _ = env.reset(seed=0)
close = np.abs(obs[0]['position'] - obs[1]['position']).max() <= 2
test_eq(bool(obs[0]['message_mask'][1]), bool(close))
test_fail(lambda: CommunicationWrapper(env.env, message_fn=lambda *a: None), contains='message_dim')

In [ ]:
# in a World: datasets record the messages that arrived (one network and channel per env)
with tempfile.TemporaryDirectory() as tmp:
    world = sm.World('MultiGrid-FindGoal-15x15-v0', num_envs=2, agents=2, size=7, num_obstacles=0, max_episode_steps=6,
                     pre_wrappers=[communication(message_keys=('position',), channel=AWGNChannel(10, 0.3))])
    assert world.envs.envs[0].unwrapped is not world.envs.envs[1].unwrapped
    world.set_policy(sm.RandomPolicy(seed=0))
    world.collect(f'{tmp}/comm.h5', episodes=4, seed=0, progress=False)
    ds = sm.HDF5Dataset(path=f'{tmp}/comm.h5', num_steps=3)
    test_eq(ds[0]['messages'].shape, (3, 2, 2, 2))                           # (T, receiver, sender, D)
    test_eq(ds[0]['message_mask'].shape, (3, 2, 2))
    assert 'comm_bits_sent' in ds.column_names

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()